# HARS Anxiety-Severity Classification — Reproducible Pipeline

This notebook provides a complete, step-by-step pipeline for score-derived HARS anxiety-severity classification, organised so that every reported table can be reproduced from a single top-to-bottom run.

**How to run:** Runtime -> Run all. In a fresh Colab runtime, run the installation cell first.

All inputs are HARS item responses only. The aggregate/total score is used solely to build the severity label and is never used as a predictive feature.

## 0. Environment Setup
Run once in a fresh Colab runtime.

In [ ]:
!pip -q install xgboost imbalanced-learn openpyxl lightgbm catboost mord

## 1. Library Imports and Reproducibility Settings

In [ ]:

# If running in a new Colab runtime, uncomment the installation line below.
# !pip -q install xgboost imbalanced-learn openpyxl

import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import entropy

from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.utils.class_weight import compute_class_weight

from imblearn.over_sampling import SMOTE

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

CLASS_NAMES = ["No Anxiety", "Mild Anxiety", "Moderate Anxiety", "Severe Anxiety", "Very Severe Anxiety"]
LABELS = np.array([0, 1, 2, 3, 4])

OUTPUT_DIR = Path("/content/revised_hars_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Study Configuration

In [ ]:

# Change this path if your Excel file is stored elsewhere.
DATA_PATH = "/content/drive/MyDrive/anxietyforCIT/Anxiety.xlsx"
SHEET_NAME = 0

# Duplicate handling options: "identifier", "exact", or "none".
# Recommended for the final analysis: "identifier" if the identifier column is reliable.
DUPLICATE_POLICY = "identifier"
IDENTIFIER_COLUMN_CANDIDATES = ["NPM", "NPM ", "Student ID", "Student Identification Number"]

# Main experiment settings.
TEST_SIZE = 0.20
N_SPLITS = 5

# Robustness settings.
MISSING_ITEM_LEVELS = [0, 2, 4, 6]
NOISE_RATES = [0.00, 0.10, 0.20, 0.30]

# To reduce runtime during early debugging, set this to False.
RUN_STACKING_ROBUSTNESS = True

N_REPEATS_CV = 10      # repeated stratified CV repeats
N_SEEDS_PROPOSED = 5   # independent splits for the proposed model (increase to 10 if time allows)


## 3. Data Loading

In [ ]:

from google.colab import drive

drive.mount('/content/drive')

raw_df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)
raw_df.columns = [str(c).strip() for c in raw_df.columns]

print("Raw dataset shape:", raw_df.shape)
print("Columns:")
for i, col in enumerate(raw_df.columns, 1):
    print(f"{i:02d}. {col}")

## 4. Identifier and Metadata Audit

In [ ]:

def find_existing_column(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower().strip() in lower_map:
            return lower_map[candidate.lower().strip()]
    return None

identifier_col = find_existing_column(raw_df, IDENTIFIER_COLUMN_CANDIDATES)
print("Identifier column detected:", identifier_col)

metadata_keywords = [
    "cap waktu", "nama", "npm", "program", "semester", "umur", "jenis kelamin", "tanggal"
]
metadata_cols = [c for c in raw_df.columns if any(k in c.lower() for k in metadata_keywords)]
print("Potential identifier/metadata columns:")
for col in metadata_cols:
    print("-", col)

## 5. Duplicate Screening

In [ ]:

df = raw_df.copy()

initial_n = len(df)
exact_duplicate_n = int(df.duplicated().sum())
identifier_duplicate_n = None

if identifier_col is not None:
    identifier_duplicate_n = int(df.duplicated(subset=[identifier_col], keep=False).sum())

print("Initial responses:", initial_n)
print("Exact duplicate rows:", exact_duplicate_n)
print("Rows involved in identifier duplicates:", identifier_duplicate_n)

if DUPLICATE_POLICY == "identifier":
    if identifier_col is None:
        raise ValueError("Identifier-based duplicate screening was selected, but no identifier column was detected.")
    df = df.drop_duplicates(subset=[identifier_col], keep="last").copy()
    print("Duplicate policy applied: identifier-based, keeping the last response per identifier.")
elif DUPLICATE_POLICY == "exact":
    df = df.drop_duplicates().copy()
    print("Duplicate policy applied: exact duplicate rows removed.")
elif DUPLICATE_POLICY == "none":
    print("Duplicate policy applied: no duplicate removal.")
else:
    raise ValueError("DUPLICATE_POLICY must be one of: 'identifier', 'exact', or 'none'.")

print("Retained responses after duplicate screening:", len(df))

## 6. HARS Item Extraction and Validation

In [ ]:

score_cols_raw = [c for c in df.columns if "pilih skor 0 - 4" in c.lower()]
print("Detected HARS score columns:", len(score_cols_raw))
for i, col in enumerate(score_cols_raw, 1):
    print(f"Q{i}: {col}")

assert len(score_cols_raw) == 14, f"Expected 14 HARS item-score columns, but found {len(score_cols_raw)}."

rename_map = {
    score_cols_raw[0]: "Q1_AnxiousMood",
    score_cols_raw[1]: "Q2_Tension",
    score_cols_raw[2]: "Q3_Fears",
    score_cols_raw[3]: "Q4_Insomnia",
    score_cols_raw[4]: "Q5_Intellectual",
    score_cols_raw[5]: "Q6_DepressedMood",
    score_cols_raw[6]: "Q7_SomaticMuscular",
    score_cols_raw[7]: "Q8_SomaticSensory",
    score_cols_raw[8]: "Q9_Cardiovascular",
    score_cols_raw[9]: "Q10_Respiratory",
    score_cols_raw[10]: "Q11_Gastrointestinal",
    score_cols_raw[11]: "Q12_Genitourinary",
    score_cols_raw[12]: "Q13_Autonomic",
    score_cols_raw[13]: "Q14_Behaviour",
}

df = df.rename(columns=rename_map)
HARS_ITEMS = list(rename_map.values())

items = df[HARS_ITEMS].apply(pd.to_numeric, errors="coerce")
missing_item_values = int(items.isna().sum().sum())
print("Missing item-score values before imputation:", missing_item_values)

# If missing values exist, median imputation is used only for item-score cleaning.
# The missing-item robustness experiment is handled separately using controlled masking.
if missing_item_values > 0:
    items = items.fillna(items.median())

item_min = float(items.min().min())
item_max = float(items.max().max())
print("Minimum item score:", item_min)
print("Maximum item score:", item_max)

if item_min < 0 or item_max > 4:
    raise ValueError("At least one HARS item score falls outside the valid range 0–4.")

items = items.astype(int)
print("Final item-response matrix shape:", items.shape)

## 7. Score-Derived Label Construction

In [ ]:

def assign_hars_severity(total_score):
    if total_score < 14:
        return 0
    elif total_score <= 20:
        return 1
    elif total_score <= 27:
        return 2
    elif total_score <= 41:
        return 3
    else:
        return 4

hars_total_score = items.sum(axis=1)
y = hars_total_score.apply(assign_hars_severity).astype(int).values

label_distribution = pd.Series(y).value_counts().sort_index().rename(index=dict(enumerate(CLASS_NAMES)))
label_distribution_df = label_distribution.reset_index()
label_distribution_df.columns = ["Severity Level", "Number of Responses"]
label_distribution_df["Percentage"] = 100 * label_distribution_df["Number of Responses"] / len(y)

print("Total score range:", int(hars_total_score.min()), int(hars_total_score.max()))
print(label_distribution_df)

label_distribution_df.to_csv(OUTPUT_DIR / "class_distribution.csv", index=False)

## 8. Internal Consistency Analysis

In [ ]:

def cronbach_alpha(dataframe):
    data = dataframe.astype(float)
    k = data.shape[1]
    item_variances = data.var(axis=0, ddof=1)
    total_variance = data.sum(axis=1).var(ddof=1)
    return (k / (k - 1)) * (1 - item_variances.sum() / total_variance)

alpha = cronbach_alpha(items)
print(f"Cronbach's alpha for the 14 HARS item responses: {alpha:.3f}")

pd.DataFrame({"Statistic": ["Cronbach's alpha"], "Value": [alpha]}).to_csv(
    OUTPUT_DIR / "reliability_analysis.csv", index=False
)

## 9. Feature Representation Design

In [ ]:

def safe_entropy(row_values):
    valid = np.asarray(row_values, dtype=float)
    valid = valid[~np.isnan(valid)]
    if len(valid) == 0:
        return np.nan
    _, counts = np.unique(valid, return_counts=True)
    return entropy(counts)


def build_feature_sets(item_frame):
    """Construct raw, pattern, distribution, combined tabular, and sequence representations."""
    item_frame = item_frame.copy().astype(float)

    raw = item_frame.copy()

    pattern = pd.DataFrame(index=item_frame.index)
    pattern["std_item"] = item_frame.std(axis=1, skipna=True)
    pattern["var_item"] = item_frame.var(axis=1, skipna=True)
    pattern["min_item"] = item_frame.min(axis=1, skipna=True)
    pattern["max_item"] = item_frame.max(axis=1, skipna=True)
    pattern["range_item"] = pattern["max_item"] - pattern["min_item"]
    pattern["iqr_item"] = item_frame.quantile(0.75, axis=1) - item_frame.quantile(0.25, axis=1)
    pattern["entropy_item"] = item_frame.apply(safe_entropy, axis=1)
    pattern["observed_item_count"] = item_frame.notna().sum(axis=1)
    pattern["missing_item_count"] = item_frame.isna().sum(axis=1)

    distribution = pd.DataFrame(index=item_frame.index)
    observed_count = item_frame.notna().sum(axis=1).replace(0, np.nan)
    for value in range(5):
        distribution[f"count_{value}"] = (item_frame == value).sum(axis=1)
        distribution[f"proportion_{value}"] = distribution[f"count_{value}"] / observed_count
    distribution["proportion_high_3_4"] = ((item_frame >= 3).sum(axis=1)) / observed_count
    distribution["proportion_low_0_1"] = ((item_frame <= 1).sum(axis=1)) / observed_count

    # Median imputation is applied to tabular descriptors only after feature construction.
    # The raw item representation uses a sentinel value for controlled missing-item experiments.
    pattern = pattern.replace([np.inf, -np.inf], np.nan).fillna(0)
    distribution = distribution.replace([np.inf, -np.inf], np.nan).fillna(0)

    combined_tabular = pd.concat([pattern, distribution], axis=1)

    raw_for_models = raw.fillna(-1.0)
    sequence = np.expand_dims(raw_for_models.values.astype(float), axis=-1)

    return {
        "raw_items": raw_for_models,
        "pattern_descriptors": pattern,
        "score_distribution_descriptors": distribution,
        "combined_tabular": combined_tabular,
        "sequence": sequence,
    }

feature_sets = build_feature_sets(items)

feature_description = []
for name, feature_object in feature_sets.items():
    if name == "sequence":
        dimension = str(feature_object.shape[1:])
    else:
        dimension = feature_object.shape[1]
    feature_description.append({"Representation": name, "Dimension": dimension})

feature_description_df = pd.DataFrame(feature_description)
print(feature_description_df)
feature_description_df.to_csv(OUTPUT_DIR / "feature_representation_summary.csv", index=False)

## 10. Train-Test Split

In [ ]:

all_indices = np.arange(len(y))
train_idx, test_idx = train_test_split(
    all_indices,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

y_train = y[train_idx]
y_test = y[test_idx]

print("Development set size:", len(train_idx))
print("Test set size:", len(test_idx))
print("Test distribution:")
print(pd.Series(y_test).value_counts().sort_index().rename(index=dict(enumerate(CLASS_NAMES))))

## 11. Evaluation Utilities

In [ ]:

def multiclass_brier_score(y_true, proba, labels=LABELS):
    y_onehot = label_binarize(y_true, classes=labels)
    return float(np.mean(np.sum((proba - y_onehot) ** 2, axis=1)))


def evaluate_predictions(model_name, y_true, y_pred, proba=None):
    result = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Macro-F1": f1_score(y_true, y_pred, average="macro"),
    }
    if proba is not None:
        result["Brier Score"] = multiclass_brier_score(y_true, proba)
    else:
        result["Brier Score"] = np.nan
    return result


def print_model_report(model_name, y_true, y_pred):
    print(f"\n=== {model_name} ===")
    print("Accuracy          :", accuracy_score(y_true, y_pred))
    print("Balanced Accuracy :", balanced_accuracy_score(y_true, y_pred))
    print("Macro-F1          :", f1_score(y_true, y_pred, average="macro"))
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))


def per_class_recall_table(prediction_dict, y_true):
    rows = []
    for model_name, y_pred in prediction_dict.items():
        recalls = recall_score(y_true, y_pred, average=None, labels=LABELS)
        for label, class_name, recall_value in zip(LABELS, CLASS_NAMES, recalls):
            rows.append({"Model": model_name, "Severity Level": class_name, "Recall": recall_value})
    return pd.DataFrame(rows)


def safe_smote(y_values):
    counts = pd.Series(y_values).value_counts()
    min_count = int(counts.min())
    if min_count <= 1:
        return None
    k_neighbors = max(1, min(5, min_count - 1))
    return SMOTE(random_state=RANDOM_STATE, k_neighbors=k_neighbors)

## 12. Base Model Definitions (RF, SVM, XGBoost, HistGB, LSTM)

In [ ]:

def make_rf():
    return RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced",
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
    )


def make_svm():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(
            C=2.0,
            kernel="rbf",
            gamma="scale",
            probability=True,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ])


def make_xgboost():
    if not XGBOOST_AVAILABLE:
        return None
    return XGBClassifier(
        n_estimators=800,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
    )


def make_histgb():
    return HistGradientBoostingClassifier(
        max_depth=6,
        learning_rate=0.05,
        max_iter=500,
        random_state=RANDOM_STATE,
    )


def build_lstm_model(input_shape=(14, 1), n_classes=5):
    # -1 is used as the masking value because 0 is a valid HARS response.
    inp = layers.Input(shape=input_shape)
    x = layers.Masking(mask_value=-1.0)(inp)
    x = layers.LSTM(32, return_sequences=False)(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(32, activation="relu")(x)
    out = layers.Dense(n_classes, activation="softmax")(x)
    model = models.Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

## 13. Additional Competitive Baselines (Ordinal and Gradient-Boosting Models)

Four additional baselines are evaluated: an **ordinal logistic regression** on the raw items (Frank & Hall decomposition), a **calibrated LightGBM**, a **monotonic gradient-boosting** model (monotone-increasing constraints on the raw items, consistent with HARS scoring), and **CatBoost**. The ordinal and monotonic models use raw items only and therefore cannot trivially reconstruct the total score.

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin

class OrdinalLogisticRegression(BaseEstimator, ClassifierMixin):
    """Frank & Hall (2001) ordinal decomposition over K-1 cumulative logits P(y > t)."""
    def __init__(self, C=1.0, random_state=RANDOM_STATE):
        self.C = C
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        self.classes_ = np.unique(y)
        self.k_ = len(self.classes_)
        self.models_ = []
        for t in self.classes_[:-1]:
            y_bin = (y > t).astype(int)
            m = Pipeline([
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(C=self.C, max_iter=2000,
                                           class_weight="balanced",
                                           random_state=self.random_state)),
            ])
            m.fit(X, y_bin)
            self.models_.append(m)
        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        gt = np.column_stack([m.predict_proba(X)[:, 1] for m in self.models_])
        probs = np.zeros((X.shape[0], self.k_))
        probs[:, 0] = 1.0 - gt[:, 0]
        for i in range(1, self.k_ - 1):
            probs[:, i] = gt[:, i - 1] - gt[:, i]
        probs[:, -1] = gt[:, -1]
        probs = np.clip(probs, 1e-9, None)
        return probs / probs.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


def make_ordinal():
    return OrdinalLogisticRegression(C=1.0)

def make_lightgbm(calibrated=True):
    from lightgbm import LGBMClassifier
    base = LGBMClassifier(n_estimators=600, learning_rate=0.05, num_leaves=31,
                          subsample=0.9, colsample_bytree=0.9,
                          class_weight="balanced", random_state=RANDOM_STATE, verbose=-1)
    if calibrated:
        return CalibratedClassifierCV(base, method="isotonic", cv=3)
    return base

def make_monotonic_gbm(n_features):
    from lightgbm import LGBMClassifier
    return LGBMClassifier(n_estimators=600, learning_rate=0.05, num_leaves=31,
                          monotone_constraints=[1] * n_features,
                          class_weight="balanced", random_state=RANDOM_STATE, verbose=-1)

def make_catboost():
    from catboost import CatBoostClassifier
    return CatBoostClassifier(iterations=600, depth=6, learning_rate=0.05,
                              loss_function="MultiClass", random_seed=RANDOM_STATE, verbose=0)

print("Additional baseline factories ready.")

## 14. Main Complete-Input Model Comparison

All learning-based models are evaluated on the same stratified test set. Tree/boosting/SVM models use the combined tabular representation; ordinal and monotonic models use raw items; the LSTM uses the ordered sequence.

In [ ]:

def fit_tabular_model(model, X_train, y_train, X_test, use_smote=False):
    if use_smote:
        smote = safe_smote(y_train)
        if smote is not None:
            X_fit, y_fit = smote.fit_resample(X_train, y_train)
        else:
            X_fit, y_fit = X_train, y_train
    else:
        X_fit, y_fit = X_train, y_train
    model.fit(X_fit, y_fit)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test) if hasattr(model, "predict_proba") else None
    return pred, proba, model

X_main = feature_sets["combined_tabular"]
S_main = feature_sets["sequence"]

X_train_main = X_main.iloc[train_idx]
X_test_main = X_main.iloc[test_idx]
S_train_main = S_main[train_idx]
S_test_main = S_main[test_idx]

main_results = []
main_predictions = {}
main_probas = {}

# Tabular models.
for model_name, model in [
    ("Random Forest", make_rf()),
    ("SVM (RBF)", make_svm()),
    ("HistGradientBoosting", make_histgb()),
]:
    pred, proba, fitted_model = fit_tabular_model(model, X_train_main, y_train, X_test_main, use_smote=False)
    main_results.append(evaluate_predictions(model_name, y_test, pred, proba))
    main_predictions[model_name] = pred
    main_probas[model_name] = proba
    print_model_report(model_name, y_test, pred)

if XGBOOST_AVAILABLE:
    pred, proba, fitted_model = fit_tabular_model(make_xgboost(), X_train_main, y_train, X_test_main, use_smote=False)
    main_results.append(evaluate_predictions("XGBoost", y_test, pred, proba))
    main_predictions["XGBoost"] = pred
    main_probas["XGBoost"] = proba
    print_model_report("XGBoost", y_test, pred)

# LSTM sequence model.
n_classes = len(LABELS)
lstm_model = build_lstm_model(input_shape=S_train_main.shape[1:], n_classes=n_classes)
class_weights_array = compute_class_weight(class_weight="balanced", classes=LABELS, y=y_train)
class_weight = {int(label): float(weight) for label, weight in zip(LABELS, class_weights_array)}

lstm_callbacks = [
    callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", patience=5, factor=0.5),
]

lstm_model.fit(
    S_train_main,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=16,
    callbacks=lstm_callbacks,
    class_weight=class_weight,
    verbose=0,
)

lstm_proba = lstm_model.predict(S_test_main, verbose=0)
lstm_pred = np.argmax(lstm_proba, axis=1)
main_results.append(evaluate_predictions("LSTM", y_test, lstm_pred, lstm_proba))
main_predictions["LSTM"] = lstm_pred
main_probas["LSTM"] = lstm_proba
print_model_report("LSTM", y_test, lstm_pred)

# --- Additional baseline models (same split) ---
X_raw_main = feature_sets["raw_items"]

# Ordinal logistic regression (raw items)
ord_pred, ord_proba, _ = fit_tabular_model(make_ordinal(),
        X_raw_main.iloc[train_idx], y_train, X_raw_main.iloc[test_idx])
main_results.append(evaluate_predictions("Ordinal LR (raw items)", y_test, ord_pred, ord_proba))
main_predictions["Ordinal LR (raw items)"] = ord_pred
main_probas["Ordinal LR (raw items)"] = ord_proba

# Calibrated LightGBM (combined tabular)
try:
    p, pr, _ = fit_tabular_model(make_lightgbm(True), X_train_main, y_train, X_test_main)
    main_results.append(evaluate_predictions("Calibrated LightGBM", y_test, p, pr))
    main_predictions["Calibrated LightGBM"] = p; main_probas["Calibrated LightGBM"] = pr
except Exception as e:
    print("LightGBM skipped:", e)

# Monotonic GBM (raw items, monotone-increasing per item)
try:
    mono = make_monotonic_gbm(X_raw_main.shape[1])
    p, pr, _ = fit_tabular_model(mono, X_raw_main.iloc[train_idx], y_train, X_raw_main.iloc[test_idx])
    main_results.append(evaluate_predictions("Monotonic GBM (raw items)", y_test, p, pr))
    main_predictions["Monotonic GBM (raw items)"] = p; main_probas["Monotonic GBM (raw items)"] = pr
except Exception as e:
    print("Monotonic GBM skipped:", e)

# CatBoost (combined tabular)
try:
    cat = make_catboost(); cat.fit(X_train_main, y_train)
    p = cat.predict(X_test_main).ravel().astype(int); pr = cat.predict_proba(X_test_main)
    main_results.append(evaluate_predictions("CatBoost", y_test, p, pr))
    main_predictions["CatBoost"] = p; main_probas["CatBoost"] = pr
except Exception as e:
    print("CatBoost skipped:", e)


## 15. Leakage-Safe Stacked Ensemble with Out-of-Fold Meta-Features

In [ ]:

def fit_stacked_ensemble(X_tabular, sequence_array, y, train_idx, test_idx, use_calibrated_tabular=True):
    X_train = X_tabular.iloc[train_idx]
    X_test = X_tabular.iloc[test_idx]
    S_train = sequence_array[train_idx]
    S_test = sequence_array[test_idx]
    y_train_local = y[train_idx]
    y_test_local = y[test_idx]

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    n_classes = len(LABELS)

    oof_rf = np.zeros((len(y_train_local), n_classes))
    oof_svm = np.zeros((len(y_train_local), n_classes))
    oof_lstm = np.zeros((len(y_train_local), n_classes))

    rf_folds, svm_folds, lstm_folds = [], [], []

    for fold, (tr, va) in enumerate(skf.split(X_train, y_train_local), 1):
        X_tr, X_va = X_train.iloc[tr], X_train.iloc[va]
        y_tr, y_va = y_train_local[tr], y_train_local[va]

        smote = safe_smote(y_tr)
        if smote is not None:
            X_tr_sm, y_tr_sm = smote.fit_resample(X_tr, y_tr)
        else:
            X_tr_sm, y_tr_sm = X_tr, y_tr

        if use_calibrated_tabular:
            rf_fold = CalibratedClassifierCV(make_rf(), method="sigmoid", cv=3)
            svm_fold = CalibratedClassifierCV(make_svm(), method="sigmoid", cv=3)
        else:
            rf_fold = make_rf()
            svm_fold = make_svm()

        rf_fold.fit(X_tr_sm, y_tr_sm)
        svm_fold.fit(X_tr_sm, y_tr_sm)

        oof_rf[va] = rf_fold.predict_proba(X_va)
        oof_svm[va] = svm_fold.predict_proba(X_va)

        rf_folds.append(rf_fold)
        svm_folds.append(svm_fold)

        S_tr, S_va = S_train[tr], S_train[va]
        class_weights_array = compute_class_weight(class_weight="balanced", classes=LABELS, y=y_tr)
        class_weight = {int(label): float(weight) for label, weight in zip(LABELS, class_weights_array)}

        lstm_fold = build_lstm_model(input_shape=S_tr.shape[1:], n_classes=n_classes)
        lstm_fold.fit(
            S_tr,
            y_tr,
            validation_data=(S_va, y_va),
            epochs=80,
            batch_size=16,
            callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)],
            class_weight=class_weight,
            verbose=0,
        )
        oof_lstm[va] = lstm_fold.predict(S_va, verbose=0)
        lstm_folds.append(lstm_fold)
        print(f"Stacking fold {fold} completed.")

    X_meta_train = np.hstack([oof_rf, oof_svm, oof_lstm])

    # Meta-learner C is selected using OOF meta-features only, not the test set.
    best_meta = None
    for c_value in [0.1, 0.3, 1.0, 3.0, 10.0]:
        candidate = LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            C=c_value,
            random_state=RANDOM_STATE,
        )
        candidate.fit(X_meta_train, y_train_local)
        oof_pred = candidate.predict(X_meta_train)
        oof_balanced = balanced_accuracy_score(y_train_local, oof_pred)
        if best_meta is None or oof_balanced > best_meta["balanced_accuracy"]:
            best_meta = {"model": candidate, "C": c_value, "balanced_accuracy": oof_balanced}

    meta_model = best_meta["model"]
    print("Selected meta-learner C:", best_meta["C"])

    rf_test_proba = np.mean([model.predict_proba(X_test) for model in rf_folds], axis=0)
    svm_test_proba = np.mean([model.predict_proba(X_test) for model in svm_folds], axis=0)
    lstm_test_proba = np.mean([model.predict(S_test, verbose=0) for model in lstm_folds], axis=0)

    X_meta_test = np.hstack([rf_test_proba, svm_test_proba, lstm_test_proba])
    stack_proba = meta_model.predict_proba(X_meta_test)
    stack_pred = meta_model.predict(X_meta_test)

    return {
        "prediction": stack_pred,
        "probability": stack_proba,
        "meta_model": meta_model,
        "base_probabilities": {
            "rf": rf_test_proba,
            "svm": svm_test_proba,
            "lstm": lstm_test_proba,
        },
    }

stacking_output = fit_stacked_ensemble(X_main, S_main, y, train_idx, test_idx, use_calibrated_tabular=True)
stack_pred = stacking_output["prediction"]
stack_proba = stacking_output["probability"]

main_results.append(evaluate_predictions("Stacking (Proposed)", y_test, stack_pred, stack_proba))
main_predictions["Stacking (Proposed)"] = stack_pred
main_probas["Stacking (Proposed)"] = stack_proba
print_model_report("Stacking (Proposed)", y_test, stack_pred)

main_results_df = pd.DataFrame(main_results)
main_results_df = main_results_df.sort_values("Balanced Accuracy", ascending=False)
print(main_results_df)
main_results_df.to_csv(OUTPUT_DIR / "main_performance_table.csv", index=False)

## 16. Deterministic Scoring and Transparent Rule References

These are **not** competitive models. Because the labels are a deterministic function of the items, a rule that simply re-sums the items reproduces the labels exactly (accuracy = 1.0). Reporting this explicitly makes clear that clean-input accuracy is not the contribution; the value of the learned models is demonstrated under constrained inputs (Section 20).

In [ ]:

reference_pred = np.array([assign_hars_severity(score) for score in hars_total_score.iloc[test_idx]])
reference_result = evaluate_predictions("Deterministic HARS Scoring Reference", y_test, reference_pred, None)
print(reference_result)
print("This reference must not be interpreted as a competing machine-learning model.")

def rule_based_prediction(item_frame):
    """Transparent rule baseline: re-sum observed items, re-apply thresholds.
    Missing items are treated as 0, which is precisely why the rule is fragile
    under missingness (see Section 20)."""
    summed = item_frame.fillna(0).sum(axis=1)
    return np.array([assign_hars_severity(s) for s in summed])

rule_pred_clean = rule_based_prediction(items.iloc[test_idx])
rule_clean = evaluate_predictions("Rule-based threshold reference", y_test, rule_pred_clean, None)
print("Transparent rule baseline on the clean test set:", rule_clean)
print("Both references reach perfect accuracy by construction and are NOT competing models.")

## 17. Feature-Representation Ablation

In [ ]:

def evaluate_tabular_representation(representation_name, X_representation):
    X_train = X_representation.iloc[train_idx]
    X_test = X_representation.iloc[test_idx]
    model = make_xgboost() if XGBOOST_AVAILABLE else make_rf()
    pred, proba, _ = fit_tabular_model(model, X_train, y_train, X_test, use_smote=False)
    return evaluate_predictions(representation_name, y_test, pred, proba)

ablation_results = []
ablation_results.append(evaluate_tabular_representation("Raw items only", feature_sets["raw_items"]))
ablation_results.append(evaluate_tabular_representation("Non-aggregate pattern descriptors only", feature_sets["pattern_descriptors"]))
ablation_results.append(evaluate_tabular_representation("Score-distribution descriptors only", feature_sets["score_distribution_descriptors"]))
ablation_results.append(evaluate_tabular_representation("Pattern + score-distribution descriptors", feature_sets["combined_tabular"]))
ablation_results.append(evaluate_predictions("Ordered sequence only (LSTM)", y_test, lstm_pred, lstm_proba))
ablation_results.append(evaluate_predictions("Stacking fusion", y_test, stack_pred, stack_proba))

ablation_df = pd.DataFrame(ablation_results).sort_values("Balanced Accuracy", ascending=False)
print(ablation_df)
ablation_df.to_csv(OUTPUT_DIR / "feature_ablation_table.csv", index=False)

## 17b. Robustness Helper Functions

Item-masking, item-perturbation, and the stacking re-fit used by the missing-item, noisy-response, and degradation analyses (Sections 18–20).

In [ ]:
def mask_items_randomly(item_frame, n_missing, seed):
    rng = np.random.default_rng(seed)
    masked = item_frame.astype(float).copy()
    if n_missing == 0:
        return masked
    n_items = masked.shape[1]
    for row_position in range(masked.shape[0]):
        cols_to_mask = rng.choice(n_items, size=min(n_missing, n_items), replace=False)
        masked.iloc[row_position, cols_to_mask] = np.nan
    return masked


def perturb_items_randomly(item_frame, noise_rate, seed):
    rng = np.random.default_rng(seed)
    perturbed = item_frame.astype(int).copy()
    if noise_rate == 0:
        return perturbed
    mask = rng.random(perturbed.shape) < noise_rate
    shifts = rng.choice([-1, 1], size=perturbed.shape)
    values = perturbed.values.copy()
    values[mask] = np.clip(values[mask] + shifts[mask], 0, 4)
    return pd.DataFrame(values, columns=perturbed.columns, index=perturbed.index)


def run_lightweight_stacking_or_xgb(item_frame, experiment_name):
    current_features = build_feature_sets(item_frame)
    current_X = current_features["combined_tabular"]
    current_S = current_features["sequence"]
    if RUN_STACKING_ROBUSTNESS:
        output = fit_stacked_ensemble(current_X, current_S, y, train_idx, test_idx,
                                      use_calibrated_tabular=True)
        return evaluate_predictions(experiment_name, y_test,
                                    output["prediction"], output["probability"])
    else:
        model = make_xgboost() if XGBOOST_AVAILABLE else make_rf()
        pred, proba, _ = fit_tabular_model(model, current_X.iloc[train_idx], y_train,
                                           current_X.iloc[test_idx], use_smote=False)
        return evaluate_predictions(experiment_name, y_test, pred, proba)

print("Robustness helper functions ready.")

## 18. Missing-Item Robustness (Proposed Model)

The zero-missing row reuses the model's stored clean-test result rather than retraining the stochastic stack, so the clean condition is numerically identical to the main comparison.

In [ ]:
missing_results = []
for n_missing in MISSING_ITEM_LEVELS:
    if n_missing == 0:
        result = evaluate_predictions("Missing items: 0", y_test, stack_pred, stack_proba)
    else:
        masked_items = mask_items_randomly(items, n_missing=n_missing, seed=RANDOM_STATE + n_missing)
        result = run_lightweight_stacking_or_xgb(masked_items, f"Missing items: {n_missing}")
    result["Missing Items per Response"] = n_missing
    missing_results.append(result)

missing_results_df = pd.DataFrame(missing_results)
print(missing_results_df)
missing_results_df.to_csv(OUTPUT_DIR / "missing_item_robustness_table.csv", index=False)

## 19. Noisy-Response Robustness (Proposed Model)

**Consistency fix:** the 0% row reuses the proposed model's stored clean-test result for the same reason as Section 18.

In [ ]:
noise_results = []
for noise_rate in NOISE_RATES:
    if noise_rate == 0:
        result = evaluate_predictions("Noisy responses: 0%", y_test, stack_pred, stack_proba)
    else:
        perturbed_items = perturb_items_randomly(items, noise_rate=noise_rate,
                                                  seed=RANDOM_STATE + int(noise_rate * 1000))
        result = run_lightweight_stacking_or_xgb(perturbed_items, f"Noisy responses: {noise_rate:.0%}")
    result["Noise Rate"] = noise_rate
    noise_results.append(result)

noise_results_df = pd.DataFrame(noise_results)
print(noise_results_df)
noise_results_df.to_csv(OUTPUT_DIR / "noisy_response_robustness_table.csv", index=False)

## 20. Rule-vs-Model Degradation under Constrained Inputs

On complete input the deterministic rule is perfect, but it collapses as items are removed or perturbed, whereas the proposed model degrades gracefully. This demonstrates that the learned model has a role **beyond** complete deterministic scoring — precisely under the incomplete/noisy conditions that occur in real screening.

In [ ]:
degradation_rows = []

# Missing-item conditions: rule recomputed, model reused from Section 18.
for row in missing_results:
    n = row["Missing Items per Response"]
    if n == 0:
        masked = items
    else:
        masked = mask_items_randomly(items, n_missing=n, seed=RANDOM_STATE + n)
    r_pred = rule_based_prediction(masked.iloc[test_idx])
    degradation_rows.append({
        "Condition": f"Missing {n} items",
        "Rule Accuracy": accuracy_score(y_test, r_pred),
        "Rule BalAcc": balanced_accuracy_score(y_test, r_pred),
        "Proposed Accuracy": row["Accuracy"],
        "Proposed BalAcc": row["Balanced Accuracy"],
    })

# Noisy conditions: rule recomputed, model reused from Section 19.
for row in noise_results:
    nr = row["Noise Rate"]
    if nr == 0:
        perturbed = items
    else:
        perturbed = perturb_items_randomly(items, noise_rate=nr, seed=RANDOM_STATE + int(nr * 1000))
    r_pred = rule_based_prediction(perturbed.iloc[test_idx])
    degradation_rows.append({
        "Condition": f"Noise {nr:.0%}",
        "Rule Accuracy": accuracy_score(y_test, r_pred),
        "Rule BalAcc": balanced_accuracy_score(y_test, r_pred),
        "Proposed Accuracy": row["Accuracy"],
        "Proposed BalAcc": row["Balanced Accuracy"],
    })

degradation_df = pd.DataFrame(degradation_rows)
print(degradation_df.round(4).to_string(index=False))
degradation_df.to_csv(OUTPUT_DIR / "rule_vs_model_degradation.csv", index=False)

## 21. Per-Class Recall and Confusion Matrices

Both the integer-count confusion matrix (for independent verification) and the normalised version are reported.

In [ ]:

recall_df = per_class_recall_table(main_predictions, y_test)
print(recall_df)
recall_df.to_csv(OUTPUT_DIR / "per_class_recall_table.csv", index=False)

# Normalised confusion matrix for the proposed model.
cm = confusion_matrix(y_test, stack_pred, labels=LABELS, normalize="true")
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, values_format=".2f", colorbar=False)
ax.set_title("Normalised Confusion Matrix: Proposed Stacking Model")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix_stacking.png", dpi=300)
plt.show()

# Confusion matrix as INTEGER COUNTS (verifiable) — overwrites the figure used in the paper
cm_counts = confusion_matrix(y_test, stack_pred, labels=LABELS)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_counts, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, values_format="d", colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix (counts): Proposed Stacking Model")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix_stacking.png", dpi=300)
plt.show()

## 22. Repeated Stratified Cross-Validation with 95% Confidence Intervals

A single 80/20 split is limited for a sample of n = 306 with only 19 very-severe cases. The lightweight (non-LSTM) competitors are therefore evaluated under repeated stratified cross-validation with mean, standard deviation, and percentile 95% confidence intervals.

In [ ]:
def repeated_cv_scores(model_factory, X, n_splits=N_SPLITS, n_repeats=N_REPEATS_CV, seed=RANDOM_STATE):
    rskf = StratifiedKFold  # placeholder import guard
    from sklearn.model_selection import RepeatedStratifiedKFold
    rkf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=seed)
    Xv = X.values if hasattr(X, "values") else X
    acc, bal, mf1 = [], [], []
    for tr_i, te_i in rkf.split(Xv, y):
        m = model_factory(); m.fit(Xv[tr_i], y[tr_i]); pred = m.predict(Xv[te_i])
        acc.append(accuracy_score(y[te_i], pred))
        bal.append(balanced_accuracy_score(y[te_i], pred))
        mf1.append(f1_score(y[te_i], pred, average="macro"))
    return {"Accuracy": acc, "Balanced Accuracy": bal, "Macro-F1": mf1}

def summarise_cv(name, rows):
    out = {"Model": name}
    for metric, vals in rows.items():
        vals = np.array(vals); lo, hi = np.percentile(vals, [2.5, 97.5])
        out[f"{metric} mean"] = round(vals.mean(), 4)
        out[f"{metric} SD"] = round(vals.std(), 4)
        out[f"{metric} CI95"] = f"[{lo:.4f}, {hi:.4f}]"
    return out

cv_specs = {"Ordinal LR (raw items)": (make_ordinal, feature_sets["raw_items"])}
try:
    from lightgbm import LGBMClassifier  # noqa
    cv_specs["Calibrated LightGBM"] = (lambda: make_lightgbm(True), feature_sets["combined_tabular"])
except Exception:
    pass

cv_summary = [summarise_cv(name, repeated_cv_scores(f, X)) for name, (f, X) in cv_specs.items()]
cv_summary_df = pd.DataFrame(cv_summary)
print(cv_summary_df.to_string(index=False))
cv_summary_df.to_csv(OUTPUT_DIR / "repeated_cv_summary.csv", index=False)

## 23. Multi-Seed Evaluation of the Proposed Model

A nested repeated CV around the LSTM stack is computationally heavy, so the full proposed pipeline is instead re-run over several independent stratified splits. The reported mean and 95% CI replace the single-split point estimate in the abstract, and quantify the run-to-run variability of the stochastic LSTM component.

In [ ]:
from sklearn.model_selection import train_test_split as _tts
seed_rows = {"Accuracy": [], "Balanced Accuracy": [], "Macro-F1": []}
all_idx = np.arange(len(y))
for s in range(N_SEEDS_PROPOSED):
    tr_s, te_s = _tts(all_idx, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE + s)
    out = fit_stacked_ensemble(feature_sets["combined_tabular"], feature_sets["sequence"],
                               y, tr_s, te_s, use_calibrated_tabular=True)
    pred = out["prediction"]
    seed_rows["Accuracy"].append(accuracy_score(y[te_s], pred))
    seed_rows["Balanced Accuracy"].append(balanced_accuracy_score(y[te_s], pred))
    seed_rows["Macro-F1"].append(f1_score(y[te_s], pred, average="macro"))
    print(f"  seed {s}: acc={seed_rows['Accuracy'][-1]:.4f}")

proposed_cv_df = pd.DataFrame([summarise_cv("Stacking (Proposed), multi-seed", seed_rows)])
print(proposed_cv_df.to_string(index=False))
proposed_cv_df.to_csv(OUTPUT_DIR / "proposed_multiseed_summary.csv", index=False)

## 24. Figures for Manuscript

In [ ]:

# Main performance comparison.
plot_df = main_results_df.set_index("Model").loc[:, ["Accuracy", "Balanced Accuracy", "Macro-F1"]]
fig, ax = plt.subplots(figsize=(10, 5))
plot_df.plot(kind="bar", ax=ax)
ax.set_ylim(0.0, 1.05)
ax.set_ylabel("Score")
ax.set_title("Overall Performance Comparison")
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "overall_performance_comparison.png", dpi=300)
plt.show()

# Brier score comparison.
brier_df = main_results_df.dropna(subset=["Brier Score"]).set_index("Model")["Brier Score"]
fig, ax = plt.subplots(figsize=(9, 5))
brier_df.plot(kind="bar", ax=ax)
ax.set_ylabel("Multiclass Brier Score")
ax.set_title("Probabilistic Calibration Comparison (Lower is Better)")
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "brier_score_comparison.png", dpi=300)
plt.show()

# Robustness plots.
if len(missing_results_df) > 0:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(missing_results_df["Missing Items per Response"], missing_results_df["Balanced Accuracy"], marker="o", label="Balanced Accuracy")
    ax.plot(missing_results_df["Missing Items per Response"], missing_results_df["Macro-F1"], marker="s", label="Macro-F1")
    ax.set_xlabel("Number of Masked Items per Response")
    ax.set_ylabel("Score")
    ax.set_title("Missing-Item Robustness")
    ax.set_ylim(0.0, 1.05)
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "missing_item_robustness.png", dpi=300)
    plt.show()

if len(noise_results_df) > 0:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(noise_results_df["Noise Rate"], noise_results_df["Balanced Accuracy"], marker="o", label="Balanced Accuracy")
    ax.plot(noise_results_df["Noise Rate"], noise_results_df["Macro-F1"], marker="s", label="Macro-F1")
    ax.set_xlabel("Item Perturbation Rate")
    ax.set_ylabel("Score")
    ax.set_title("Noisy-Response Robustness")
    ax.set_ylim(0.0, 1.05)
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "noisy_response_robustness.png", dpi=300)
    plt.show()

## 25. Export Summary

In [ ]:

print("Exported files:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print("-", path)

print("The deterministic HARS scoring rule is reported only as a label-generating reference, not as a competing predictive model.")
print("Feature ablation, missing-item robustness, and noisy-response robustness analyses are used to evaluate the role of item-response representations beyond complete clean scoring conditions.")

In [ ]:
import shutil
shutil.copytree('/content/revised_hars_outputs',
                '/content/drive/MyDrive/anxietyforCIT/revised_hars_outputs',
                dirs_exist_ok=True)
print("Semua hasil baru sudah disalin ke Google Drive: anxietyforCIT/revised_hars_outputs")